In [1]:
import pandas as pd
import numpy as np
import os
import pickle
from typing import Dict, Any

# --- 경로 설정 및 모델 저장 위치 수정 ---
# VENDOR_ANALYSIS_PATH의 경로가 Jupyter Notebook의 위치에 따라 상대적으로 설정됨
VENDOR_ANALYSIS_PATH = "../../output/problem2_vendor/problem2_vendor_analysis_base.csv"
OUT_DIR = "../../output/model" # 모델 아티팩트 저장 폴더
MODEL_FILE_NAME = "model_ver2.pkl" # 모델 파일명 수정
os.makedirs(OUT_DIR, exist_ok=True) # 폴더가 없으면 생성


# --- 모델 하이퍼파라미터 정의 ---
Z_SCORE_THRESHOLD = 2.0
MOST_SENSITIVE_GICS_SECTORS = [] # 과제 3 제외

# 분석 및 출력에 사용할 모든 수익률 컬럼
ALL_RETURN_COLUMNS = ['return_post_1d', 'return_post_2d', 'return_post_5d', 'return_post_10d', 'return_post_20d']


# --- 2. 기본 규칙 기반 의사 결정 함수 ---
def get_investment_decision_basic(df: pd.DataFrame) -> pd.DataFrame:
    """
    ARIMA Surprise Z-Score (±2.0)만을 기준으로 매수/매도 결정을 내리는 기본 모델.
    (이 함수는 Z-Score 임계값 결정만 담당합니다.)
    """
    df_result = df.copy()
    
    # NOTE: 최종 모델에서는 이 Z-Score 임계값을 훈련 데이터 통계치로 계산해야 함.
    is_buy_signal = df_result['surprise_z'] > Z_SCORE_THRESHOLD
    is_sell_signal = df_result['surprise_z'] < -Z_SCORE_THRESHOLD
    
    conditions = [
        is_buy_signal,
        is_sell_signal,
        True
    ]
    
    choices = [
        'BUY', 
        'SELL', 
        'HOLD'
    ]
    
    df_result['decision'] = np.select(conditions, choices, default='HOLD')
    return df_result


# --- 3. 모델 실행, 검증 및 저장 ---
print("-> 1. 데이터 로드 및 모델 Version 2 실행...")

try:
    # 데이터 로드 (장기 수익률 컬럼 추가)
    df_analysis = pd.read_csv(
        VENDOR_ANALYSIS_PATH, 
        # 모든 장기 수익률 컬럼을 usecols에 포함
        usecols=['symbol', 'date', 'surprise_z'] + ALL_RETURN_COLUMNS,
        dtype={col: np.float32 for col in ['surprise_z'] + ALL_RETURN_COLUMNS}
    ).dropna(subset=['surprise_z', 'return_post_1d']) # return_post_1d가 있는 행만 사용
    
    # 모델 실행
    df_decision = get_investment_decision_basic(df_analysis)
    
    # 시그널 발생 행만 필터링
    df_signals = df_decision[df_decision['decision'].isin(['BUY', 'SELL'])].copy()
    
    if df_signals.empty:
        print("분석: BUY/SELL 시그널이 발생하지 않아 검증을 수행할 수 없습니다.")
        exit()

    # 4. 모델 성능 검증 (장기 수익률 요약)
    # 모든 장기 수익률 컬럼을 집계에 포함
    df_summary = df_signals.groupby('decision')[ALL_RETURN_COLUMNS].mean().mul(100).round(4)
    
    # 컬럼 이름 변경: Avg_Return_Post_1D (%), Avg_Return_Post_20D (%) 등으로 변경
    column_rename_map = {
        col: f"Avg_Return_Post_{col.replace('return_post_', '').replace('d', 'D')} (%)" 
        for col in ALL_RETURN_COLUMNS
    }
    df_summary = df_summary.rename(columns=column_rename_map)

    # 5. 모델 규칙 저장 (.pkl)
    model_rules = {
        'model_name': 'Basic_ZScore_Classifier',
        'z_threshold': Z_SCORE_THRESHOLD,
        'sensitive_sectors': MOST_SENSITIVE_GICS_SECTORS, 
        'version': 'v2.0 (Long-Term Returns Added)' # 버전 업데이트
    }
    
    model_path = os.path.join(OUT_DIR, MODEL_FILE_NAME)
    with open(model_path, 'wb') as f:
        pickle.dump(model_rules, f)
        
    print(f"\n[OK] 모델 규칙 저장 완료: {model_path}")
    
    # 6. 결과 출력
    print("\n" + "="*85)
    print("과제 4: 모델 Version 2 장기 수익률 검증 결과 (산업 필터 제외)")
    print("="*85)
    print(df_summary.to_markdown())
    print("="*85)

except FileNotFoundError as e:
    print(f"❌ 오류: 필요한 파일을 찾을 수 없습니다. 경로를 확인하세요: {e}")
except Exception as e:
    print(f"❌ 오류: 데이터 처리 중 예상치 못한 오류 발생: {e}")

-> 1. 데이터 로드 및 모델 Version 2 실행...

[OK] 모델 규칙 저장 완료: ../../output/model/model_ver2.pkl

과제 4: 모델 Version 2 장기 수익률 검증 결과 (산업 필터 제외)
| decision   |   Avg_Return_Post_1D (%) |   Avg_Return_Post_2D (%) |   Avg_Return_Post_5D (%) |   Avg_Return_Post_10D (%) |   Avg_Return_Post_20D (%) |
|:-----------|-------------------------:|-------------------------:|-------------------------:|--------------------------:|--------------------------:|
| BUY        |                   0.2718 |                   0.8935 |                   1.5915 |                    2.5234 |                    2.3723 |
| SELL       |                   0.3046 |                   0.7187 |                   0.1945 |                    0.7846 |                   -0.0842 |
